# Lab04.4 — Contours and Object Tracking

**Hardware:** Zybo Legacy (PYNQ 3.0.1), Logitech C270, OpenCV 4.5.4.

**Objective:** track the **largest connected colored region** in webcam images, compute its contour area and centroid, and classify the position as **LEFT / CENTER / RIGHT / LOST**.

This notebook extends the experimentally tested HSV + RGB dominance segmentation from Lab04.3. A marker with white labels can produce multiple contours: we track the largest colored region, **not necessarily the entire marker**. No additional sensors, overlays or installations are required.

## 1. Setup and reusable functions

Run this setup cell once. The default target is **GREEN**. Change `TARGET_COLOR` to `RED` or `BLUE` to follow a different marker. The camera is released automatically after each acquisition session.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
TARGET_COLOR = "GREEN"             # RED, GREEN, or BLUE
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
WARMUP_FRAMES = 30
MIN_AREA = 150                   # Area of a connected color region in pixels^2

COLORS = {
    "RED": {"ranges": [((0, 100, 50), (10, 255, 255)),
                       ((170, 100, 50), (179, 255, 255))],
            "margin": 35, "box": (0, 0, 255)},
    "GREEN": {"ranges": [((35, 45, 35), (90, 255, 255))],
              "margin": 10, "box": (0, 255, 0)},
    "BLUE": {"ranges": [((95, 80, 50), (130, 255, 255))],
             "margin": 35, "box": (255, 0, 0)}
}

@contextmanager
def camera_session():
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open camera: " + CAMERA_DEVICE)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
        # Discard initial frames to stabilize automatic exposure.
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None:
                raise RuntimeError("Camera warm-up frame failed.")
            time.sleep(0.05)
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpeg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpeg.tobytes()))

def make_color_mask(frame, color_name):
    if color_name not in COLORS:
        raise ValueError("TARGET_COLOR must be RED, GREEN, or BLUE")
    cfg = COLORS[color_name]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    b, g, r = [c.astype(np.int16) for c in cv2.split(frame)]
    hsv_mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lower, upper in cfg["ranges"]:
        hsv_mask = cv2.bitwise_or(hsv_mask, cv2.inRange(hsv, lower, upper))
    m = cfg["margin"]
    if color_name == "RED":
        dominant = (r > g + m) & (r > b + m) & (r > 80)
    elif color_name == "GREEN":
        dominant = (g > r + m) & (g > b + m) & (g > 50)
    else:
        dominant = (b > r + m) & (b > g + m) & (b > 80)
    mask = cv2.bitwise_and(hsv_mask, dominant.astype(np.uint8) * 255)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    )
    return mask

def track_largest_region(frame, color_name=TARGET_COLOR):
    # Track only the largest connected region above the area threshold.
    mask = make_color_mask(frame, color_name)
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    candidates = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]
    annotated = frame.copy()
    h, w = frame.shape[:2]
    # Divide the image into three equal horizontal zones.
    cv2.line(annotated, (w // 3, 0), (w // 3, h - 1), (160, 160, 160), 1)
    cv2.line(annotated, (2 * w // 3, 0), (2 * w // 3, h - 1), (160, 160, 160), 1)
    if not candidates:
        state, centroid, area = "LOST", None, 0.0
    else:
        contour = max(candidates, key=cv2.contourArea)
        area = cv2.contourArea(contour)
        moments = cv2.moments(contour)
        x, y, bw, bh = cv2.boundingRect(contour)
        if moments["m00"] != 0:
            cx = int(moments["m10"] / moments["m00"])
            cy = int(moments["m01"] / moments["m00"])
        else:
            cx, cy = x + bw // 2, y + bh // 2
        centroid = (cx, cy)
        state = "LEFT" if cx < w / 3 else "RIGHT" if cx >= 2 * w / 3 else "CENTER"
        color = COLORS[color_name]["box"]
        cv2.rectangle(annotated, (x, y), (x + bw, y + bh), color, 2)
        cv2.circle(annotated, centroid, 5, color, -1)
        cv2.putText(annotated, "{} ({},{}) area={:.0f}".format(
            color_name, cx, cy, area), (max(5, x), max(18, y - 10)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.47, color, 1)
    cv2.putText(annotated, "Position: " + state, (10, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return annotated, mask, state, centroid, area

print("OpenCV:", cv2.__version__, "| Target:", TARGET_COLOR)

## 2. Single-frame verification

Hold a **green marker** in view, run this cell and inspect the image, binary mask, bounding rectangle, centroid and state. Move it into LEFT, CENTER or RIGHT and rerun. Remove the marker to check **LOST**.

This test avoids a continuous loop until the basic detection is confirmed.

In [ ]:
with camera_session() as cap:
    ok, frame = cap.read()
    if not ok or frame is None:
        raise RuntimeError("No camera frame captured.")

result, mask, state, centroid, area = track_largest_region(frame, TARGET_COLOR)
print("Position:", state, "| Centroid:", centroid, "| Region area:", round(area, 1))
print("Original | Binary mask | Tracking result")
panel = np.hstack([
    cv2.resize(img, (320, 240))
    for img in (frame, cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR), result)
])
show_bgr(panel)

## 3. Track across successive frames

After the single-frame test succeeds, move the marker horizontally in front of the camera. The loop ends after **60 frames or 12 seconds**, whichever is reached first. To reduce Jupyter output and CPU overhead, the display updates only every fifth frame.

The elapsed-time check occurs between reads: it cannot interrupt a blocked USB driver call. The final frame and last position are displayed when processing finishes.

In [ ]:
MAX_FRAMES = 60
MAX_SECONDS = 12
DISPLAY_EVERY = 5
processed = 0
last_state = "LOST"

with camera_session() as cap:
    start = time.monotonic()
    for i in range(MAX_FRAMES):
        if time.monotonic() - start >= MAX_SECONDS:
            break
        ok, frame = cap.read()
        if not ok or frame is None:
            print("WARNING: Camera frame read failed.")
            break
        result, mask, last_state, centroid, area = track_largest_region(
            frame, TARGET_COLOR
        )
        processed += 1
        if i % DISPLAY_EVERY == 0:
            show_bgr(cv2.resize(result, (480, 360)), replace=True)

# Display the last frame without replacing the completion message.
print("Processed frames:", processed, "| Last position:", last_state)
if processed:
    show_bgr(cv2.resize(result, (480, 360)))

## Student tasks

1. Record the detected centroid `(cx, cy)` while the colored marker is in LEFT, CENTER and RIGHT.
2. Remove the marker and explain why the state becomes **LOST**.
3. Change `TARGET_COLOR` from `GREEN` to `RED` or `BLUE`, rerun the setup cell and repeat the test.
4. Move the marker closer to the camera; explain why contour area changes.
5. Explain why only the **largest colored region** is tracked, especially when a marker contains white labels.

**Acceptance check:** the camera reopens after repeated runs; position changes as the marker moves; LOST appears when no valid colored region remains; the camera is released on exit.

**Limitations:** the algorithm classifies colored image regions, not unique physical objects. Multiple objects of the same color can cause the selected region to change. It does not perform persistent object identification.